# **History Aware Answer Generation**

In [8]:
from langchain_chroma import Chroma

# represents user input, sets rules/behavior for the model, represents the model’s response
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings

### **Configurations**

In [9]:
db_path = "db/chroma_db"

### **Load same embedding model**

In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="saved_models/all-mpnet-base-v2", # all-MiniLM-L12-v2, all-MiniLM-L6-v2, all-mpnet-base-v2
    model_kwargs={"device":"cpu"},
    encode_kwargs={"normalize_embeddings":True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### **Connect Vector Database**

In [11]:
db = Chroma(
    persist_directory=db_path,
    embedding_function=embedding_model,
    collection_metadata={"hnsw:space":"cosine"}
)

### **LLM**

In [12]:
llm = ChatOllama(model="llama3")

In [14]:
# Store chat coversations as msgs
chat_sitory = []

In [47]:
def ask_question(user_question):
    print(f"### User asked : {user_question} ***")

    # ================= step 01 (question updating)=================
    
    # If chat history is not empty contine hostory aware conversation
    # first create new msg that contain all previous chats including  
    #    1 - SystemMessage(at startup), 
    #    ALL human + llm messages
    #    user new msg with prefix new quesrtion: in type HumanMessage
    # passed to llm and get new question
    if chat_sitory:
        question_prompt = [
            # Define the rules/ isntruction to LLM
            SystemMessage(content="Given the chat history, rewrite the new question to be standalone and searchable. Just return the rewritten question.")
        ] + chat_sitory + [
            HumanMessage(content=f"new quesrtion: {user_question}")
        ]

        print(f"### New question genrating...")
        # print(f"### prompt : {question_prompt}")
        result = llm.invoke(question_prompt)
        search_question = result.content.strip()
        print(f"### Searching for : {search_question} ***")
        
    # if no hoistory initialize it (first question)
    else:
        search_question= user_question

    # ================= step 02 (relevent doc finding) =================
    
    # let find relevant docs using new updaetd question that genrated by llm in step 1
    retriever = db.as_retriever(search_kwargs={"k": 3})
    relevant_docs = retriever.invoke(search_question)
    print("### Relevant docs found")


    # ================= step 03 (final user msg) =================
    
    combined_docs = "\n".join([f"- {doc.page_content}" for doc in relevant_docs])

    combined_input = f"""
        User Message:
        {user_question}
    
        Documents:
        {combined_docs}
        
        Instruction:
        Generate a helpful banking response based on the intent. 
        If you can't find helpfull answer in docs say I'm wasn't able to find any information please contact support.
    """

    #  ================= step 04 (final prompt) =================

    final_prompt = [
        SystemMessage(content="You are a helpful assistant that answer questions based on provided documents and conversation hostory.")
    ] + chat_sitory + [
        HumanMessage(content=combined_input)
    ]

    print(f"### Generating answer...")
    # print(f"### prompt : {final_prompt} ***")
    result = llm.invoke(final_prompt)
    answer= result.content

    #  ================= step 05 (history updating ) =================

    # Save the conversation/ History updating
    chat_sitory.append(HumanMessage(content=user_question))
    chat_sitory.append(AIMessage(content=answer))

    print("-"*100)
    print(f"Answer: {answer} ***")
    print("-"*100)
    
    return answer

In [48]:
def start_chat():
    print("Ask me question : type 'quit' to exit")

    while True:
        question= input("AsK : ")
        if question.lower() == 'quit':
            print("Thank you")
            break
        ask_question(question)

In [49]:
chat_sitory= []
start_chat()

Ask me question : type 'quit' to exit


AsK :  explain nvidea


### User asked : explain nvidea ***
### Relevant docs found
### Generating answer...
----------------------------------------------------------------------------------------------------
Answer: Nvidia! Let me help you understand more about this tech giant.

From what I've gathered from the provided documents, Nvidia is a leading technology company that specializes in designing and manufacturing graphics processing units (GPUs) for gaming, professional visualization, and artificial intelligence. They also develop high-performance computing hardware for datacenter and cloud computing applications.

Some interesting points to note:

1. As mentioned in document 242 by TechSpot, Nvidia has faced criticism for its practices, such as prioritizing profits over ethics and sustainability.
2. In document 175, VentureBeat highlights Nvidia's Maxine AI technology, which can create deepfakes (i.e., manipulated audio or video recordings) and raises concerns about bias in video calls.
3. According to 

AsK :  its main income


### User asked : its main income ***
### New question genrating...
### Searching for : What is NVIDIA's main source of income? ***
### Relevant docs found
### Generating answer...
----------------------------------------------------------------------------------------------------
Answer: Based on the provided documents, Nvidia's main income comes from:

* GPUs for video gaming and creative workloads: As of 2025, Nvidia controlled more than 80% of the market for GPUs used in training and deploying AI models. [9][10][11]
* Professional GPUs for edge computing, scientific research, and industrial applications: The company's product lines include professional GPUs for various industries.
* Gaming hardware and services: Nvidia has expanded into gaming hardware and services, with products like laptops and virtual workstations that enable remote work.

Additionally, the company reported sales of $3.87 billion in Q2 2020, a 50% rise from the same period in 2019. [7] As of May 2023, Nvidia cros

AsK :  quit


Thank you
